In [ ]:
import numpy as np
import time

BASE = '/home/xilinx/jupyter_notebooks/fpga-prosthetic-poc/'

p = np.load(BASE + 'svm_params.npz')
scaler_mean = p['scaler_mean']
scaler_std  = p['scaler_std']
sv          = p['support_vectors']
dual_coef   = p['dual_coef']
intercept   = p['intercept']
gamma       = float(p['gamma'][0])
classes     = p['classes']
n_support   = p['n_support']

n_classes = len(classes)
sv_start = np.concatenate([[0], np.cumsum(n_support[:-1])])

print('로드 완료')
print(f'  클래스 수: {n_classes}, 서포트 벡터: {sv.shape[0]}')

In [ ]:
def svm_predict(X):
    # 1. 정규화
    X_scaled = (X - scaler_mean) / scaler_std

    # 2. RBF 커널: K[n, m] = exp(-gamma * ||x_n - sv_m||^2)
    diff = X_scaled[:, np.newaxis, :] - sv[np.newaxis, :, :]  # (N, n_sv, feat)
    K = np.exp(-gamma * np.sum(diff ** 2, axis=2))             # (N, n_sv)

    # 3. OvO 투표
    votes = np.zeros((len(X), n_classes), dtype=np.int32)
    pair_idx = 0
    for i in range(n_classes):
        for j in range(i + 1, n_classes):
            si, ei = sv_start[i], sv_start[i] + n_support[i]
            sj, ej = sv_start[j], sv_start[j] + n_support[j]

            # 클래스 i의 SV 계수: dual_coef[j-1, si:ei]
            # 클래스 j의 SV 계수: dual_coef[i,   sj:ej]
            d = (K[:, si:ei] @ dual_coef[j-1, si:ei]
               + K[:, sj:ej] @ dual_coef[i,   sj:ej]
               + intercept[pair_idx])

            votes[:, i] += (d > 0).astype(np.int32)
            votes[:, j] += (d <= 0).astype(np.int32)
            pair_idx += 1

    return classes[np.argmax(votes, axis=1)]

# 테스트
X_test = np.random.randn(1, 96).astype(np.float32)
print('예측 결과:', svm_predict(X_test))

In [ ]:
# 워밍업
for _ in range(10):
    svm_predict(X_test)

# 지연시간 측정 (100회)
latencies = []
for _ in range(100):
    start = time.perf_counter()
    svm_predict(X_test)
    end = time.perf_counter()
    latencies.append((end - start) * 1000)

print('[PYNQ ARM baseline]')
print(f'  평균 지연시간: {np.mean(latencies):.3f} ms')
print(f'  표준편차:     {np.std(latencies):.3f} ms')
print(f'  최소:         {np.min(latencies):.3f} ms')
print(f'  최대:         {np.max(latencies):.3f} ms')